# SASHIMI-SI: independent scientific comparison

A is corrected upstream `e17d3664dac677b604fd4ff02fb2af105a6937fa`. B adds an independent 50-digit analytic NFW inverse, evaluates only the selected cross-section branch, and computes the effective cross section at 65-digit precision. The product evaluates the same defining expression as a positive integral to avoid cancellation. The product uses a separate root solver and the common executor. This notebook checks migration agreement, not grid convergence or SIDM calibration. The total-cross-section one-power denominator was adopted after independent review and angular-integration checks; the separate effective cross section used here is unchanged.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from sashimi_si import SubhaloProperties
reference_dir = Path("tests/references")
if not reference_dir.exists():
    reference_dir = Path("../tests/references")
provenance = json.loads((reference_dir/"B-accurate.json").read_text())
parameters = provenance["calculation"]["parameters"]
model = SubhaloProperties()
result = model.subhalo_properties_calc(**parameters)
catalogs = model.subhalo_catalogs_calc(**parameters)
reference = np.load(reference_dir/"B-accurate.npz")
metrics = []
for i, actual in enumerate(result):
    expected = reference[f"tuple_{i}"]
    if i in (25, 26):
        np.testing.assert_array_equal(actual, expected)
    else:
        np.testing.assert_allclose(actual, expected, rtol=5e-12, atol=1e-300)
    metrics.append(float(np.max(np.abs(np.asarray(actual,float)-expected)/np.maximum(np.abs(expected),1e-300))))
print("All 27 fields agree. Maximum relative difference:", max(metrics))
print("Reference source:", provenance["source_revision"])
print("Product specification:", catalogs["sidm"].metadata["calculation_specification"])
bundle = reference_dir.parents[1]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
edges = np.geomspace(1e4, 1e7, 10)
for state, index in [("cdm_reference", 23), ("sidm", 24)]:
    catalog = catalogs[state]
    counts, _ = np.histogram(catalog.columns["m_bound"], edges, weights=catalog.weight_final)
    ref_counts, _ = np.histogram(reference["tuple_11"], edges, weights=reference[f"tuple_{index}"])
    np.testing.assert_allclose(counts, ref_counts, rtol=5e-12)
    axes[0].step(np.sqrt(edges[:-1]*edges[1:]), counts/np.diff(np.log(edges)), where="mid", label=state)
axes[0].set(xscale="log", xlabel="Bound mass [Msun]", ylabel="dN/dln M")
axes[0].legend()
axes[1].semilogy(np.arange(27), np.maximum(metrics,1e-16), "o")
axes[1].set(xlabel="Historical tuple field", ylabel="Max relative B/C difference")
fig.tight_layout()
plt.show()

In [ ]:
formation_dir = reference_dir.parent/"formation_reference"
formation_provenance = json.loads((formation_dir/"B-accurate-formation.json").read_text())
formed = model.subhalo_catalogs_calc(**formation_provenance["calculation"]["parameters"])
valid = formed["sidm"].columns["valid_accretion"]
assert valid.size == 168 and valid.sum() == 164
for state in formed.values():
    assert np.all(state.weight_final[~valid] == 0)
    assert all(np.all(np.isfinite(v)) for v in state.columns.values())
print("Formation boundary: 164 physical nodes, 4 unformed nodes with explicit validity flags.")

Both state masks, their independent base weights and representative mass functions are checked. Historical inverse interpolation error is not included in the B/C tolerance. Host mass, redshift, both quadrature orders, solver, threshold and SI parameters are fixed. Larger-grid convergence and strong-interaction calibration remain separate release evidence.

## Saved one-variable resolution sweep

The hash-verified summary comes from complete saved catalogs, with original
source SHAs and any failure/warning records retained. Baseline M0=1e12 Msun,
zmax=3, N_ma=16, dz=.25, N_herm=3, N_hermNa=8, ct_th=.77 and pert2_shanks
are fixed unless the plotted axis changes. Accretion M200 spans 1e6–1e10 Msun.
The finest grid is a comparator, not continuum truth or a universal error bound.

In [ ]:
import hashlib
import matplotlib.pyplot as plt
science_dir = bundle / "validation/science"
summary_file = science_dir / "convergence-summary.json"
provenance = json.loads((science_dir / "provenance.json").read_text())
assert hashlib.sha256(summary_file.read_bytes()).hexdigest() == provenance["summary_sha256"]
summary = json.loads(summary_file.read_text())
state = "default" if summary["variant"] == "sashimi-c" else "sidm"
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for axis, parameter, base_value in zip(axes, ["N_ma", "dz"], [16, .25]):
    selected = [(base_value, summary["rows"]["baseline"])]
    selected += [(row["parameters"][parameter], row) for label, row in summary["rows"].items() if label.startswith(parameter + "-")]
    selected.sort(key=lambda pair: pair[0], reverse=parameter == "dz")
    values = np.array([row["states"][state]["metrics"]["bound_mass_fraction"] for _, row in selected])
    axis.semilogx([x for x, _ in selected], (values / values[-1] - 1) * 100, "o-")
    axis.set(xlabel="Mass nodes N_ma" if parameter == "N_ma" else "Redshift step dz", ylabel="Bound mass fraction difference from finest [%]")
    if parameter == "dz": axis.invert_xaxis()
    axis.grid(alpha=.2)
fig.suptitle("Finite-grid sensitivity at fixed reduced settings")
fig.tight_layout()
plt.show()
for check in summary["final_refinements"]:
    print(check["before"], "→", check["after"], "bound mass change [%]", format(check["relative_percent"]["bound_mass_fraction"], ".6g"))

The last mass-node and redshift refinements still change bound mass fractions
by about 0.50% and 0.75–0.77%, respectively. At the coarse baseline, switching
pert2_shanks to direct ODE changes bound mass by about -3.82%; tightening ODE
tolerances has a much smaller effect. These distinctions remain visible and do
not change the product defaults. See `docs/resolution-and-states.md` for the
full table, original EPS failure and separate post-fix successful controls.

## SIDM states and aligned CDM check

This separate sweep fixes zmax=2 and varies only sigma0. Weak interactions
approach the model's own CDM reference; strong interactions remove additional
survivors. Raw C and SI differ in density normalization. Only the explicitly
aligned diagnostic view matches their radii/densities to roundoff; product
calibrations were not changed.

In [ ]:
states = json.loads((science_dir / "states-and-cdm.json").read_text())
weak = states["rows"][0]
assert weak["cdm_count"] == weak["sidm_count"]
assert max(weak["weak_differences"].values()) < 5e-8
fig, ax = plt.subplots(figsize=(6, 3.5))
sigma = [row["sigma0"] for row in states["rows"]]
ax.semilogx(sigma, [row["cdm_count"] for row in states["rows"]], "o--", label="CDM reference")
ax.semilogx(sigma, [row["sidm_count"] for row in states["rows"]], "s-", label="SIDM")
ax.set(xlabel="sigma0 [cm²/g]", ylabel="Expected surviving count", title="Specified weak and strong interaction cases")
ax.legend(); fig.tight_layout(); plt.show()
print("Weak maximum profile discrepancy:", max(weak["weak_differences"].values()))
print("Aligned C/SI comparison:", weak["matched_nodes_C"][-1]["differences"])